# fastanki

> Python tools for Anki

fastanki reads and writes Anki's collection format and speaks the AnkiWeb sync protocol directly, in Python. There's no dependency on the Anki application or its Rust library: your cards live in a small sqlite file of fastanki's own, and reach your desktop and phone the same way any Anki client's changes do, by syncing through AnkiWeb. Media files aren't synced yet, so create text-only cards.

## Usage

### Installation

Install latest from [pypi][pypi]


```sh
$ pip install fastanki
```

[pypi]: https://pypi.org/project/fastanki/

### Documentation

In [ ]:
from fastanki import *
import os, tempfile

In [ ]:
#| hide
os.environ['FASTANKI_DIR'] = tempfile.mkdtemp()

## Functional API

`add_card` lets you create a new card with a single function call. Just pass your field values as keyword arguments. By default it uses the Basic note type and Default deck, but you can specify any model, deck, or tags you like.

In [ ]:
notezh = add_card(fields={'Front':'你好','Back':'hello'})

`find_cards` searches your collection and returns a list of `Card` objects. Criteria are keyword arguments, all optional and combined with AND:

- `deck='Spanish'` matches that deck and its subdecks
- `tag='vocab'` matches a tag
- `added_days=7` matches cards added in the last week
- `is_due=True` matches cards due for review
- Any other keyword is a field name, matched as a case-insensitive substring: `Front='hello'`
- `where="n.id=?", args=(nid,)` drops through to SQL over `notes n` joined with `cards c`

In [ ]:
cards = find_cards(deck='Default')
cards

[Card(1784904710443, nid=1784904710443, due=1, ivl=0, queue=0)]

In [ ]:
cards[0]

<div class="prose" markdown="1">

Card 1784904710443 (nid: 1784904710443): new #1

</div>

In [ ]:
find_card_ids(deck='Default')

[1784904710443]

`find_notes` takes the same criteria and returns one `Note` per matching note, where `find_cards` may return several cards for a note (a Cloze note generates one card per cloze number, for instance).

In [ ]:
notes = find_notes(fields={'Back':'hello'})
notes

[Note(1784904710443, Front='你好', Back='hello', tags=[])]

In [ ]:
note = notes[0]
note

<div class="prose" markdown="1">

**Front**: 你好 | **Back**: hello | 🏷 -

</div>

In [ ]:
find_note_ids(fields={'Back':'hello'})

[1784904710443]

`update_note` modifies an existing note's fields and/or tags. Pass either a `Note` object or a note ID, along with any fields you want to change as keyword arguments. For tags:
- `tags=['a','b']` — replaces all tags
- `add_tags='newtag'` — adds without removing existing tags

In [ ]:
update_note(note, Back="updated answer", tags='testtag')

<div class="prose" markdown="1">

**Front**: 你好 | **Back**: updated answer | 🏷 testtag

</div>

In [ ]:
update_note(note, add_tags='moretagz')

<div class="prose" markdown="1">

**Front**: 你好 | **Back**: updated answer | 🏷 testtag moretagz

</div>

In [ ]:
get_note(note.id)

<div class="prose" markdown="1">

**Front**: 你好 | **Back**: updated answer | 🏷 testtag moretagz

</div>

In [ ]:
del_note([notezh, note])

2

Reviewing is the same loop every Anki client runs: `next_card` picks what to study, `answer_buttons` reports each button's delay (for display), `answer_card` grades it (1=Again, 2=Hard, 3=Good, 4=Easy). New cards walk the learning steps and graduate to review cards; scheduling follows the collection's own settings (SM-2 or FSRS, whichever your other clients use) and every answer is a review-log entry that `sync` carries to all your devices. A fresh card settles in one sitting, because learning cards are served a little early once nothing else is due:

In [ ]:
nid = add_fb_card('¿cómo estás?', 'how are you?')
def delay(s): return f'{round(s/86400)}d' if s>=86400 else f'{round(s/60)}m'
while (card := next_card()):
    labels = ' · '.join(f'{l} {delay(s)}' for l,(_,s) in zip(('Again','Hard','Good','Easy'), answer_buttons(card.id)))
    print(f"{get_note(card.nid)['Front']}  [{labels}]")
    answer_card(card.id, 3)

¿cómo estás?  [Again 1m · Hard 6m · Good 10m · Easy 4d]


¿cómo estás?  [Again 1m · Hard 10m · Good 1d · Easy 4d]


In [ ]:
find_cards(fields={'Front':'cómo'})[0]

<div class="prose" markdown="1">

Card 1784904710676 (nid: 1784904710676): review, ivl 1d, due day 1

</div>

In [ ]:
#| hide
del_note(nid)

1

`sync` connects to AnkiWeb: pass your credentials the first time, and they're saved (as a host key, not your password) for later calls. The first sync of a fresh collection is a full download of your existing AnkiWeb collection; after that, syncs exchange deltas in both directions. fastanki will never replace a non-empty server collection without an explicit `upload=True`.

In [ ]:
#| eval: false
sync(user=os.environ['ANKI_USER'], passw=os.environ['ANKI_PASS'])  # first time
sync()  # after that

KeyError: 'ANKI_USER'

## Tool use

In [ ]:
anki_tools()

&`[add_card, add_fb_card, add_cloze_card, find_notes, find_note_ids, find_cards, find_card_ids, get_note, del_note, update_fb_note, next_card, answer_buttons, answer_card, due_counts, sync]`


Here are the available tools: 
&`[add_card, add_fb_card, add_cloze_card, find_notes, find_note_ids, find_cards, find_card_ids, get_note, del_note, update_fb_note, sync]`.

Try to find all my notes. List the IDs and contents you see.

Delete them.

Try finding all notes again.

Try adding a note of your choice using `add_fb_card` and tell me the id.

Try finding all notes again.

OK try `get_note` with it.

Delete it now.

OK create, update, and verify a note now.

Try the various find ones that we haven't done yet.

Sure. Delete that note, then sync.

## OO API

The functional API opens and closes the collection on every call. For a batch of work, `Collection` keeps it open, and everything the functions above do is a method here.

In [ ]:
col = Collection.open()
col.path.name

'collection.anki2'

In [ ]:
col.notetypes(), col.decks()

(['Basic', 'Cloze'], ['Default'])

In [ ]:
n = col.add(Front='adiós', Back='goodbye', deck='Spanish::Vocab', tags=['spanish'])
n

<div class="prose" markdown="1">

**Front**: adiós | **Back**: goodbye | 🏷 spanish

</div>

In [ ]:
col.due_counts('Spanish')

(1, 0, 0)

In [ ]:
col.find_notes(deck='Spanish')

[Note(1784904710932, Front='adiós', Back='goodbye', tags=['spanish'])]

In [ ]:
col.remove_deck('Spanish')
col.close()

`Collection` is also a context manager, so a one-shot batch reads naturally:

In [ ]:
with Collection.open() as c: c.add(Front='hola', Back='hello')